# Dataset to be used
The link to the dataset: https://data.cityofnewyork.us/Public-Safety/Emergency-Response-Incidents/pasr-j7fb/data_preview
This data is provided by the NYC Emergency Management (NYCEM) that demonstrates the type and address of emergency incidents.


# Analysis question: 
Between 2016 and 2024, are emergency response incidents in New York City more frequently recorded in the outer boroughs (Bronx, Brooklyn, Queens, and Staten Island) than in Manhattan, and does this pattern differ by incident type?

# Columns that will be used:
Borough

Incident Type

Creation Date

# Hypothesis
Emergency response incidents are more frequently recorded in the outer boroughs than in Manhattan, especially for medical and structural incident types.


# Loading data

In [2]:
import pandas as pd
import plotly.express as px

In [3]:
df = pd.read_csv("Emergency_Response_Incidents_20251130.csv")
df.head()

,Incident Type,Location,Borough,Creation Date,Closed Date,Latitude,Longitude
0,Utility-Water Main,136-17 72 Avenue,Queens,2017 Jan 16 01:13:38 PM,NaN,40.714004,-73.829989
1,Structural-Sidewalk Collapse,927 Broadway,Manhattan,2016 Oct 29 12:13:31 PM,NaN,40.714422,-74.006076
2,Utility-Other,NaN,Manhattan,2016 Nov 22 08:53:17 AM,NaN,NaN,NaN
3,Administration-Other,Seagirt Blvd & Beach 9 Street,Queens,2016 Nov 14 03:53:54 PM,NaN,40.714004,-73.829989
4,Law Enforcement-Other,NaN,Brooklyn,2016 Oct 29 05:35:28 PM,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11750 entries, 0 to 11749
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Incident Type  11750 non-null  object 
 1   Location       10873 non-null  object 
 2   Borough        11750 non-null  object 
 3   Creation Date  11750 non-null  object 
 4   Closed Date    8642 non-null   object 
 5   Latitude       10125 non-null  float64
 6   Longitude      10125 non-null  float64
dtypes: float64(2), object(5)
memory usage: 642.7+ KB


# Data Cleaning
I am making sure the three columns I'm using are correctly labelled, transforming dates to datetime and removing any missing or invalid fields.
After that, I'm going to sort the data.

In [5]:
df.columns

Index(['Incident Type', 'Location', 'Borough', 'Creation Date', 'Closed Date',
       'Latitude', 'Longitude'],
      dtype='object')

In [10]:
# Only keeping these columns
df = df[["Borough", "Incident Type", "Creation Date"]].copy()

#Cleaning up text columns (strip spaces and standardize cases)
df["Borough"] = df["Borough"].astype(str).str.strip().str.upper()
df["Incident Type"] = df["Incident Type"].astype(str).str.strip()

# Converting Creation Date to datetime
df["Creation Date"] = pd.to_datetime(df["Creation Date"], errors="coerce")

# Removing rows with missing information
df = df.dropna(subset=["Borough", "Incident Type", "Creation Date"])

# Keeping the 5 valid boroughs
valid_boroughs = ["BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND"]
df = df[df["Borough"].isin(valid_boroughs)]

# First sort by creation date to ensure chronological order, then by borough
df = df.sort_values(by=["Creation Date", "Borough"]).reset_index(drop=True)

# Double check
df.head()



,Borough,Incident Type,Creation Date
0,BRONX,Fire-4th Alarm,2011-05-04 06:03:58
1,BRONX,Utility-Transformer Fire,2011-05-04 06:09:36
2,MANHATTAN,Utility-Electric Feeder Cable,2011-05-04 07:46:18
3,MANHATTAN,Structural-Construction Accident,2011-05-04 09:29:07
4,BRONX,Fire-2nd Alarm,2011-05-04 14:44:43


In [11]:
# Double checking the boroughs are all sorted correctly
df["Borough"].value_counts(dropna=False)

Borough
MANHATTAN        3658
BROOKLYN         3445
QUEENS           2056
BRONX            1619
STATEN ISLAND     580
Name: count, dtype: int64

In [15]:
# Filter year to between 2016 and 2024
df["year"] = df["Creation Date"].dt.year
df = df[(df["year"] >= 2016) & (df["year"] <= 2024)]
df.shape


(7656, 4)

In [16]:
# Calculate incident counts for period 2016-2024
incident_counts = df["Borough"].value_counts().reset_index()
incident_counts.columns = ["Borough", "Count"]
incident_counts


,Borough,Count
0,BROOKLYN,2477
1,MANHATTAN,2262
2,QUEENS,1397
3,BRONX,1152
4,STATEN ISLAND,368


# Data Visualization: Bar Chart


In [17]:
fig = px.bar(
    incident_counts,
    x="Borough",
    y="Count",
    title="Emergency Response Incidents by Borough (2016–2024)",
    labels={"Count": "Number of Incidents"},
    text="Count"
)

fig.update_layout(
    xaxis_title="Borough",
    yaxis_title="Number of Incidents",
    title_x=0.5
)

fig.show()


This shows that Brooklyn has the most number of emergency response incidents of 2477 between year 2016 to 2024. Manhattan has the second most number of emergency response incidents, with Staten Island with the least with 368 incidents. 

In [19]:
# Finding the top 5 most common incident types between 2016–2024
top_types = df["Incident Type"].value_counts().nlargest(5).index

df_top = df[df["Incident Type"].isin(top_types)]

# Then grouping the incidents type by borough
by_borough_type = (
    df_top.groupby(["Borough", "Incident Type"])
          .size()
          .reset_index(name="Count")
)



In [20]:
fig = px.bar(
    by_borough_type,
    x="Borough",
    y="Count",
    facet_col="Incident Type",
    facet_col_wrap=3,  # wrap into multiple rows if needed
    title="Emergency Incidents by Borough and Incident Type (2016–2024)",
    labels={"Count": "Number of Incidents"}
)

fig.update_layout(title_x=0.5)
fig.show()


This data visualization allows us to compare borough patterns within each incident type. From the data, the pattern differs by incident type. Brooklyn experiences the highest frequency of multi-alarm fire incidents, suggesting larger residential density and more buildings with fire-risk activity. Power outage incidents cluster heavily in Brooklyn, possibly due to older infrastructure, denser residential areas, or greater dependence on above-ground utilities. Water main issues are more prevalent in Manhattan—possibly linked to older underground pipes and high-pressure water systems serving large commercial and residential buildings. Suspicious package calls concentrate in Manhattan, which could be because of high tourist volume, key transportation hubs, high-value commercial and government targets, and greater reporting activity. Thus, the relationship between borough and incident frequency changes depending on the incident type.